# L4 34 — arena-context fidelity diagnostic

Tests the existing Qwen 3B link on 256 frozen deployment snapshots from the completed IPD run. It compares the action distribution induced by readable text with trained, shuffled, random, zero, and exact token-embedding representations.

This is a diagnostic, not another arena experiment. It is checkpointed to Drive and safe to resume.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = '5094b6542d93d1a680a180054d0a3df2e85943ef'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
JOB_ID = 'faithful-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
SAMPLES = 256
JOB_DIR = pathlib.Path('/content/drive/MyDrive/rival-arena-l4') / JOB_ID
MATCHES = JOB_DIR/'arena_ipd_confirmatory_v3/matches.jsonl'
assert (JOB_DIR/'faithful_link.pt').exists(), 'Missing trained link in Drive'
assert MATCHES.exists(), 'Missing confirmatory matches in Drive'
print('Snapshots:', SAMPLES, '| source:', MATCHES)

In [ ]:
command = [
    'python', 'scripts/diagnose_arena_fidelity.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--matches', MATCHES,
    '--samples', str(SAMPLES),
]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected output: `MyDrive/rival-arena-l4/faithful-qwen3b-t4-001/arena_context_fidelity_v1/`. The report's gate requires the trained representation to have lower text-to-action KL than both shuffled and random controls.